# 💼 Machine Learning Task 3 (2026): Resume & Candidate Screening System
### **Future Interns Internship Program**
**Author**: nithi-2008  
**Repository**: [https://github.com/nithi-2008/FUTURE_ML_03.git](https://github.com/nithi-2008/FUTURE_ML_03.git)  

---

## 📌 1. Project Background & Objective
Hiring managers receive hundreds of resumes for a single job opening. Manually screening resumes is slow, inconsistent, and prone to recruiter fatigue.

In this project, we design an intelligent **Decision-Support Natural Language Processing (NLP) system** that:
1. Parses and extracts text from unstructured resumes (PDF, DOCX, TXT).
2. Preprocesses text (cleaning, stopword removal, lemmatization, section extraction).
3. Extracts domain skills and technical proficiencies using **spaCy** phrase matching and taxonomy-based canonical mapping.
4. Compares candidates against target Job Descriptions using a hybrid scoring model (TF-IDF Cosine Similarity + Weighted Skill Matching + Experience Alignment).
5. Ranks candidates and assigns actionable tiers (`Highly Recommended`, `Potential Fit`, `Low Fit`).
6. Pinpoints exact skill gaps and formulates tailored interview screening questions for HR managers.

## 🛠️ 2. Environment Setup & Library Imports

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# NLP & ML libraries
import spacy
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Append source directory
sys.path.insert(0, os.path.abspath('..'))

from src.text_preprocessor import TextPreprocessor
from src.skill_extractor import SkillExtractor
from src.matcher import ResumeMatcher
from src.ranker import CandidateRanker
from src.explainer import ScreeningExplainer
from src.document_reader import DocumentReader

print("✅ All modules loaded successfully!")

## 📊 3. Exploratory Data Analysis (EDA) on Candidate Pool
We inspect the distribution of candidates across diverse engineering and data domains.

In [ ]:
data_path = os.path.join('..', 'data', 'resumes_dataset.csv')
df = pd.read_csv(data_path)
print(f"Total resumes loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(3)

In [ ]:
plt.figure(figsize=(10, 4))
sns.countplot(data=df, y='primary_category', palette='viridis')
plt.title('Candidate Distribution by Domain Category')
plt.xlabel('Number of Resumes')
plt.ylabel('Domain')
plt.show()

## 🧹 4. Text Preprocessing & Cleaning Pipeline
Demonstrating noise removal, regex preservation of technical tokens (e.g., C++, .NET, CI/CD), and lemmatization.

In [ ]:
preprocessor = TextPreprocessor()
sample_raw = df.iloc[0]['resume_text'][:400]
print("--- RAW RESUME SNIPPET ---")
print(sample_raw)

cleaned = preprocessor.clean_text(sample_raw)
print("\n--- CLEANED TEXT ---")
print(cleaned)

tokens = preprocessor.tokenize_and_lemmatize(sample_raw)
print("\n--- LEMMATIZED TOKENS (Sample 15) ---")
print(tokens[:15])

## 🧠 5. NLP Skill Extraction (spaCy + Taxonomy)
We extract canonical skills and group them into domains (Cloud, AI/ML, Languages, Databases).

In [ ]:
extractor = SkillExtractor()
extracted = extractor.extract_skills(df.iloc[0]['resume_text'])
print(f"Extracted {extracted['total_skills_count']} Unique Skills:")
print(extracted['found_skills'])

print("\nSkills Grouped by Domain:")
for cat, sks in extracted['skills_by_category'].items():
    print(f"  • {cat}: {', '.join(sks)}")

## 🎯 6. Target Job Description Loading & Parsing
Loading the target Senior Machine Learning Engineer benchmark role.

In [ ]:
with open(os.path.join('..', 'data', 'job_descriptions.json'), 'r') as f:
    jds = json.load(f)

target_jd = jds[0]  # Senior Machine Learning Engineer
print(f"Target Role: {target_jd['title']}")
print(f"Required Experience: {target_jd['experience_years']} Years")
print(f"Required Skills: {target_jd['required_skills']}")
print(f"Preferred Skills: {target_jd['preferred_skills']}")

## ⚙️ 7. Similarity & Hybrid Role Fit Scoring
Mathematical formula combining:
$$\text{Composite Fit} = (w_{\text{skill}} \cdot \text{Skill Match}) + (w_{\text{semantic}} \cdot \text{TF-IDF Cosine Sim}) + (w_{\text{exp}} \cdot \text{Experience Fit})$$

In [ ]:
matcher = ResumeMatcher(skill_weight=0.50, semantic_weight=0.35, experience_weight=0.15)

# Screen first 10 candidates
results = []
for idx, row in df.iloc[:10].iterrows():
    res = matcher.match_resume(row['resume_text'], target_jd, filename=f"{row['candidate_name']}.txt")
    results.append(res)

ranker = CandidateRanker()
ranked_candidates = ranker.rank_candidates(results)

ranked_df = pd.DataFrame(ranked_candidates)[['rank', 'candidate_name', 'composite_score', 'skill_score', 'semantic_score', 'tier', 'verdict']]
ranked_df

## 🔍 8. Explainable AI (XAI) & Skill Gap Analysis
Drilling down into the #1 ranked candidate vs candidates with skill gaps.

In [ ]:
explainer = ScreeningExplainer()
top_cand = ranked_candidates[0]
print(explainer.generate_recruiter_report(top_cand))

print("\n--- Suggested Interview Questions ---")
questions = explainer.generate_interview_questions(top_cand)
for q in questions:
    print(f"• [{q['type']}] {q['question']}")

## 📈 9. Conclusions & Next Steps
- The hybrid scoring architecture successfully prevents false positives caused by generic keyword stuffing.
- spaCy phrase matching with canonical mapping ensures accurate identification of synonyms (e.g. K8s = Kubernetes, Sklearn = Scikit-Learn).
- The explainable decision framework gives non-technical recruiters clear, justified recommendations.